# De búsqueda vectorial a RAG básico

En este cuaderno primero veremos la recuperación local: documentos, *chunks*, embeddings y búsqueda vectorial. Sólo después pediremos a OpenRouter que redacte una respuesta con los fragmentos encontrados.

## La pequeña biblioteca del curso

Usaremos `learning_ai_rag`, código Python incluido en `packages/learning_ai_rag/`. No es un framework: sus piezas son deliberadamente pequeñas y visibles. Leeremos documentos, los dividiremos, crearemos embeddings, buscaremos en memoria y, al final, opcionalmente generaremos una respuesta con la evidencia encontrada.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / 'packages').exists():
    root = root.parent

sys.path.insert(0, str(root / 'packages' / 'learning_ai_rag'))

from learning_ai_rag import (
    InMemoryVectorStore,
    OpenRouterAnswerGenerator,
    RAGPipeline,
    SentenceTransformerEmbedder,
    load_markdown_documents,
)
from learning_ai_rag.ingest import chunk_documents


## 1. Conocer los documentos

Los Markdown de `datos-demo/knowledge` son la única fuente de este ejercicio. Antes de buscar, conviene mirar qué títulos y políticas hay disponibles.

In [ ]:
documents = load_markdown_documents(root / 'datos-demo' / 'knowledge')
for document in documents:
    print(f'- {document.source}: {document.title}')

print('\nFragmento de la primera fuente:\n')
print(documents[0].text[:500])


## 2. Partir el texto en chunks

Un chunk es un pasaje que podremos citar. El solapamiento conserva algo de contexto entre pasajes vecinos.

In [ ]:
chunks = chunk_documents(documents, chunk_size_words=120, overlap_words=30)
print(f'Documentos: {len(documents)}')
print(f'Chunks: {len(chunks)}\n')
print(f'[{chunks[0].chunk_id}] {chunks[0].source}\n{chunks[0].text}')


## 3. Convertir texto en embeddings locales

`SentenceTransformerEmbedder` usa `paraphrase-multilingual-MiniLM-L12-v2`, un modelo local multilingüe. Convierte textos y preguntas en vectores de 384 números que podemos comparar por significado.

In [ ]:
embedder = SentenceTransformerEmbedder()
print(f'Modelo: {embedder.model_name}')
print(f'Dimensiones por embedding: {embedder.dimension}')
print(embedder.encode([chunks[0].text])[:, :8])


## 4. Guardar y buscar en memoria

`InMemoryVectorStore` conserva chunks y embeddings sólo mientras el cuaderno está en marcha. Es un almacén temporal en memoria: no persiste nada y permite observar la búsqueda vectorial sin añadir infraestructura.

In [ ]:
store = InMemoryVectorStore.from_chunks(chunks, embedder)
pipeline = RAGPipeline(store, embedder)
question = '¿Qué garantía tiene un producto reacondicionado?'
evidence_answer = pipeline.answer(question)

print(f'Pregunta: {question}\n')
print('Evidencia recuperada:\n')
print(evidence_answer.answer)
print('\nCitas:')
for citation in evidence_answer.citations:
    print(f'- {citation.chunk.source} · {citation.chunk.chunk_id} ({citation.score:.3f})')


## 5. Añadir generación: ahora sí, RAG

La recuperación anterior ya nos ha dado el contexto. Si defines `OPENROUTER_API_KEY` en `.env` o en tu terminal, la siguiente celda pasa ese contexto a OpenRouter. El modelo debe responder en español usando sólo esos chunks y citarlos con su identificador.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(root / '.env')
if not os.environ.get('OPENROUTER_API_KEY'):
    print('Define OPENROUTER_API_KEY para ejecutar esta celda. La búsqueda local ya ha funcionado.')
else:
    generator = OpenRouterAnswerGenerator()
    rag_answer = pipeline.answer_with_generator(question, generator)
    print('Respuesta RAG:\n')
    print(rag_answer.answer)
    print('\nCitas controladas por la aplicación:')
    for citation in rag_answer.citations:
        print(f'- {citation.chunk.source} · {citation.chunk.chunk_id}')


## 6. Control de abstención

Si ningún fragmento supera el umbral de similitud, el pipeline se abstiene antes de llamar al modelo. Prueba «¿Cuál es el número de tarjeta de crédito del cliente?». Con el modelo y el corpus del lab, esta pregunta queda por debajo del umbral. La formulación importa: «¿Cuál es el número de tarjeta del cliente?» puede recuperar políticas aunque no contengan ese dato. Un umbral de similitud no garantiza que haya una respuesta.

In [ ]:
unsupported = pipeline.answer('¿Cuál es el número de tarjeta de crédito del cliente?')
print(unsupported.answer)
print(f'Abstención: {unsupported.abstained}')
